In [2]:
class MedicalExpertSystem:
    def __init__(self):
        # Knowledge Base: Rules definition
        # Rule 1: IF Fever = Yes AND Cough = Yes THEN Flu
        # Rule 2: IF Flu THEN Medicine Required
        self.rules = [
            {
                "conditions": {"fever": "yes", "cough": "yes"},
                "conclusion": "flu",
                "description": "IF Fever = Yes AND Cough = Yes THEN Flu"
            },
            {
                "conditions": {"flu": "yes"},
                "conclusion": "medicine_required",
                "description": "IF Flu THEN Medicine Required"
            }
        ]

    def forward_chaining(self, initial_facts):
        """
        Forward Chaining: Data-driven reasoning.
        Starts with known facts and repeatedly applies rules until no new facts can be derived.
        """
        facts = dict(initial_facts)
        inferred = True
        derived_steps = []

        while inferred:
            inferred = False
            for rule in self.rules:
                # Check if all conditions for the rule are satisfied by current facts
                conditions_met = True
                for cond_key, cond_val in rule["conditions"].items():
                    if facts.get(cond_key, "").lower() != str(cond_val).lower():
                        conditions_met = False
                        break

                # If conditions are met and conclusion is not already in facts
                if conditions_met and rule["conclusion"] not in facts:
                    facts[rule["conclusion"]] = "yes"
                    derived_steps.append(f"Rule Applied: {rule['description']} --> Derived Fact: {rule['conclusion'].upper()} = YES")
                    inferred = True

        return facts, derived_steps

    def backward_chaining(self, goal, known_facts):
        """
        Backward Chaining: Goal-driven reasoning.
        Starts with a goal and works backward to see if supporting facts/rules exist.
        """
        # If the goal is already a known fact
        if known_facts.get(goal, "").lower() == "yes":
            return True, [f"Goal '{goal}' is already a verified fact."]

        # Search for rules that conclude the goal
        relevant_rules = [r for r in self.rules if r["conclusion"] == goal]
        if not relevant_rules:
            return False, [f"No rules found to conclude goal '{goal}'."]

        for rule in relevant_rules:
            proof_steps = [f"Checking rule: {rule['description']} to prove goal '{goal}'"]
            all_conditions_proven = True

            for cond_key in rule["conditions"]:
                # Recursively try to prove sub-goals / conditions
                sub_proven, sub_steps = self.backward_chaining(cond_key, known_facts)
                proof_steps.extend(sub_steps)
                if not sub_proven:
                    all_conditions_proven = False
                    break

            if all_conditions_proven:
                proof_steps.append(f"Goal '{goal}' successfully proven!")
                return True, proof_steps

        return False, [f"Could not prove goal '{goal}' with current facts."]

    def run_demonstration(self):
        print("=== EXPERIMENT 9: Medical Diagnosis Expert System ===")
        print("Knowledge Base Rules:")
        for r in self.rules:
            print(f" - {r['description']}")
        print("=" * 60)

        # Predefined Sample Test Case
        sample_facts = {"fever": "yes", "cough": "yes"}
        print(f"\n[Test Case 1] Running Forward Chaining with Initial Facts: {sample_facts}")
        final_facts, steps = self.forward_chaining(sample_facts)
        print("Derived Facts & Steps:")
        for step in steps:
            print(f"  * {step}")
        print(f"Final Knowledge Base State: {final_facts}")

        print("\n" + "-" * 60)
        goal = "medicine_required"
        print(f"[Test Case 2] Running Backward Chaining for Goal: '{goal}'")
        proven, bc_steps = self.backward_chaining(goal, sample_facts)
        for step in bc_steps:
            print(f"  * {step}")
        print(f"Goal Result: {'PROVEN' if proven else 'NOT PROVEN'}")

if __name__ == "__main__":
    system = MedicalExpertSystem()
    system.run_demonstration()

    print("\n" + "=" * 60)
    print("--- Interactive Medical Diagnosis Evaluation ---")
    while True:
        choice = input("\nDo you want to test custom symptoms? (y/n): ").strip().lower()
        if choice != 'y':
            break

        fever_input = input("Does the patient have a Fever? (yes/no): ").strip().lower()
        cough_input = input("Does the patient have a Cough? (yes/no): ").strip().lower()

        user_facts = {"fever": fever_input, "cough": cough_input}

        print("\n--- Forward Chaining Result ---")
        fc_facts, fc_steps = system.forward_chaining(user_facts)
        if fc_steps:
            for s in fc_steps:
                print(f"  * {s}")
        else:
            print("  * No rules triggered based on symptoms.")
        print(f"Diagnosis Result: {fc_facts}")

        print("\n--- Backward Chaining Result (Checking Goal: 'medicine_required') ---")
        proven, bc_steps = system.backward_chaining("medicine_required", user_facts)
        for s in bc_steps:
            print(f"  * {s}")
        print(f"Medicine Required Status: {'YES' if proven else 'NO'}")

=== EXPERIMENT 9: Medical Diagnosis Expert System ===
Knowledge Base Rules:
 - IF Fever = Yes AND Cough = Yes THEN Flu
 - IF Flu THEN Medicine Required

[Test Case 1] Running Forward Chaining with Initial Facts: {'fever': 'yes', 'cough': 'yes'}
Derived Facts & Steps:
  * Rule Applied: IF Fever = Yes AND Cough = Yes THEN Flu --> Derived Fact: FLU = YES
  * Rule Applied: IF Flu THEN Medicine Required --> Derived Fact: MEDICINE_REQUIRED = YES
Final Knowledge Base State: {'fever': 'yes', 'cough': 'yes', 'flu': 'yes', 'medicine_required': 'yes'}

------------------------------------------------------------
[Test Case 2] Running Backward Chaining for Goal: 'medicine_required'
  * Checking rule: IF Flu THEN Medicine Required to prove goal 'medicine_required'
  * Checking rule: IF Fever = Yes AND Cough = Yes THEN Flu to prove goal 'flu'
  * Goal 'fever' is already a verified fact.
  * Goal 'cough' is already a verified fact.
  * Goal 'flu' successfully proven!
  * Goal 'medicine_required' succ

In [ ]:
class AgricultureAdvisoryExpertSystem:
    def __init__(self):
        # Knowledge Base: Rules definition for Agricultural Advisory
        # Rule 1: IF Moisture < 30 THEN Turn On Water = Yes
        # Rule 2: IF Soil Type == Sandy AND Nitrogen < 50 THEN Add Fertilizer = NPK
        # Rule 3: IF Turn On Water == Yes AND Add Fertilizer == NPK THEN Crop Action Required = High Priority
        self.rules = [
            {
                "id": "R1",
                "conditions": lambda facts: facts.get("moisture", 100) < 30,
                "conclusion": "turn_on_water",
                "value": "yes",
                "description": "IF Moisture < 30% THEN Turn On Water = YES"
            },
            {
                "id": "R2",
                "conditions": lambda facts: facts.get("soil_type", "").lower() == "sandy" and facts.get("nitrogen", 100) < 50,
                "conclusion": "add_fertilizer",
                "value": "npk",
                "description": "IF Soil Type = Sandy AND Nitrogen < 50 THEN Add Fertilizer = NPK"
            },
            {
                "id": "R3",
                "conditions": lambda facts: facts.get("turn_on_water") == "yes" and facts.get("add_fertilizer") == "npk",
                "conclusion": "crop_action_required",
                "value": "high priority",
                "description": "IF Turn On Water = YES AND Add Fertilizer = NPK THEN Crop Action Required = HIGH PRIORITY"
            }
        ]

    def forward_chaining(self, initial_facts):
        """
        Forward Chaining: Data-driven reasoning starting with farm sensors/facts
        and repeatedly applying rules until no new facts can be derived.
        """
        facts = dict(initial_facts)
        inferred = True
        derived_steps = []

        while inferred:
            inferred = False
            for rule in self.rules:
                # Check if rule conditions are met
                try:
                    conditions_met = rule["conditions"](facts)
                except Exception:
                    conditions_met = False

                # If conditions met and conclusion not already set with this value
                if conditions_met and facts.get(rule["conclusion"]) != rule["value"]:
                    facts[rule["conclusion"]] = rule["value"]
                    derived_steps.append(f"Rule {rule['id']} Applied: {rule['description']} --> Derived Fact: {rule['conclusion'].upper()} = {str(rule['value']).upper()}")
                    inferred = True

        return facts, derived_steps

    def run_demonstration(self):
        print("=== EXPERIMENT 9: Agriculture Advisory Expert System ===")
        print("Knowledge Base Rules:")
        for r in self.rules:
            print(f" - [{r['id']}] {r['description']}")
        print("=" * 65)

        # Predefined Sample Test Cases
        sample_farms = [
            {"farm_name": "Farm Alpha", "moisture": 25, "soil_type": "sandy", "nitrogen": 40},
            {"farm_name": "Farm Beta", "moisture": 60, "soil_type": "loam", "nitrogen": 80}
        ]

        for farm in sample_farms:
            print(f"\n[Test Case] Evaluating {farm['farm_name']} with Sensors: Moisture={farm['moisture']}%, Soil={farm['soil_type']}, Nitrogen={farm['nitrogen']}")
            final_facts, steps = self.forward_chaining(farm)
            print("Derived Recommendations & Inference Steps:")
            if steps:
                for step in steps:
                    print(f"  * {step}")
            else:
                print("  * No rules triggered. Optimal conditions.")
            print(f"Final Farm Advisory State: {final_facts}")
            print("-" * 65)

if __name__ == "__main__":
    system = AgricultureAdvisoryExpertSystem()
    system.run_demonstration()

    print("\n" + "=" * 65)
    print("--- Interactive Agriculture Advisory Evaluation ---")
    while True:
        choice = input("\nDo you want to evaluate custom farm sensor data? (y/n): ").strip().lower()
        if choice != 'y':
            break

        farm_name = input("Enter Farm/Field Name: ").strip()
        try:
            moisture = float(input("Enter Soil Moisture Level (% e.g., 25): "))
            soil_type = input("Enter Soil Type (e.g., Sandy / Loam / Clay): ").strip()
            nitrogen = float(input("Enter Nitrogen Level (mg/kg e.g., 40): "))
        except ValueError:
            print("Invalid numeric input! Please enter proper numbers for moisture and nitrogen.")
            continue

        user_facts = {
            "farm_name": farm_name,
            "moisture": moisture,
            "soil_type": soil_type,
            "nitrogen": nitrogen
        }

        print(f"\n--- Forward Chaining Advisory Result for {farm_name} ---")
        fc_facts, fc_steps = system.forward_chaining(user_facts)
        if fc_steps:
            for s in fc_steps:
                print(f"  * {s}")
        else:
            print("  * No rules triggered. Soil condition is optimal.")

        print("\nFinal Decision Summary:")
        print(f" - Irrigation Needed: {'YES (Turn on water)' if fc_facts.get('turn_on_water') == 'yes' else 'NO'}")
        print(f" - Fertilizer Advice: {str(fc_facts.get('add_fertilizer', 'None')).upper()}")
        print(f" - Action Priority: {str(fc_facts.get('crop_action_required', 'Normal')).upper()}")

=== EXPERIMENT 9: Agriculture Advisory Expert System ===
Knowledge Base Rules:
 - [R1] IF Moisture < 30% THEN Turn On Water = YES
 - [R2] IF Soil Type = Sandy AND Nitrogen < 50 THEN Add Fertilizer = NPK
 - [R3] IF Turn On Water = YES AND Add Fertilizer = NPK THEN Crop Action Required = HIGH PRIORITY

[Test Case] Evaluating Farm Alpha with Sensors: Moisture=25%, Soil=sandy, Nitrogen=40
Derived Recommendations & Inference Steps:
  * Rule R1 Applied: IF Moisture < 30% THEN Turn On Water = YES --> Derived Fact: TURN_ON_WATER = YES
  * Rule R2 Applied: IF Soil Type = Sandy AND Nitrogen < 50 THEN Add Fertilizer = NPK --> Derived Fact: ADD_FERTILIZER = NPK
  * Rule R3 Applied: IF Turn On Water = YES AND Add Fertilizer = NPK THEN Crop Action Required = HIGH PRIORITY --> Derived Fact: CROP_ACTION_REQUIRED = HIGH PRIORITY
Final Farm Advisory State: {'farm_name': 'Farm Alpha', 'moisture': 25, 'soil_type': 'sandy', 'nitrogen': 40, 'turn_on_water': 'yes', 'add_fertilizer': 'npk', 'crop_action_requi